# Pipeline A — Human baseline (v2): executed verification notebook
**What runs in this kernel:** a fresh 5-fold CV (StratifiedKFold 5, seed 42, original best params, early stopping 100) on the cached
`features/features_A.parquet` (307,511 customers × 241 features), plus live predictions from the saved `models/model_A.pkl`.
**What is loaded, not regenerated:** the feature matrix (built by `src/features_a.py` via `run_all.py A`) and the final model.
Fresh CV models are ephemeral; no artifact is modified. Scores are raw, uncalibrated LightGBM probabilities.

In [1]:
import sys, os, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('BFSI_THREADS', '8')
import numpy as np, pandas as pd, joblib, pyarrow.parquet as pq
from src import config as C, train
from sklearn.metrics import roc_auc_score
RES = json.loads((C.OUT_DIR/'final_results.json').read_text())
print('ROOT', ROOT, '| packaged run finished', RES['run']['finished_at_local'], '| status', RES['execution_status'])

ROOT /workspace/bfsi-mtp/BFSI_MTP_Demo | packaged run finished 2026-11-11 23:11:45 Asia/Kolkata (UTC+05:30) | status complete


In [2]:
# PROVENANCE of the cached feature matrices (LOADED, not regenerated here)
for P in ['A']:
    print(P, 'features_%s.parquet sha256' % P, RES['artifacts_sha256']['features'][P][:16], '| model pkl sha256', RES['artifacts_sha256']['models'][P]['pkl'][:16])
print('input CSV sha256 recorded for', len(RES['data']['sha256']), 'files; generation log: outputs/run_log.txt')

A features_A.parquet sha256 accff9ad9dbdad58 | model pkl sha256 9c7322be68d95ffb
input CSV sha256 recorded for 9 files; generation log: outputs/run_log.txt


## Full rebuild (NOT executed in this notebook)
The feature matrices were generated by `python run_all.py A` (logged in `outputs/run_log.txt`).
To regenerate from raw Kaggle data run in a terminal: `python run_all.py A` (then `python run_all.py audit` and `python run_all.py finalize`).

## Fresh 5-fold CV (trained here)

In [3]:
# FRESH 5-fold CV executed in THIS kernel on the cached features_A.parquet (ephemeral models, nothing saved)
X = pq.read_table(C.FEAT_DIR/'features_A.parquet').to_pandas()
sc = pd.read_parquet(C.OUT_DIR/'scored_table.parquet', columns=['SK_ID_CURR','TARGET','oof_A','pred_A'])
assert np.array_equal(X['SK_ID_CURR'].to_numpy(), sc['SK_ID_CURR'].to_numpy()) and len(X) == 307511
ids = X.pop('SK_ID_CURR'); y = sc['TARGET'].astype(int)
t0 = time.time(); oof, m, _ = train.run_cv(X, y, 'A')
print(f'fresh CV time {time.time()-t0:.0f}s')
ref = RES['pipelines']['A']
print('fresh OOF AUC', m['oof_auc'], '| packaged', ref['oof_auc'], '| fresh folds', m['fold_aucs'])
print('max |fresh OOF - packaged oof_A| =', float(np.max(np.abs(oof - sc['oof_A'].to_numpy()))))

  A fold 1 AUC=0.78014 trees=375 t=9s


  A fold 2 AUC=0.78828 trees=374 t=17s


  A fold 3 AUC=0.78220 trees=375 t=25s


  A fold 4 AUC=0.78730 trees=373 t=34s


  A fold 5 AUC=0.78025 trees=372 t=43s
  A OOF AUC=0.78362 KS=0.42897 cv_std=0.00349
fresh CV time 43s
fresh OOF AUC 0.78362 | packaged 0.78362 | fresh folds [0.78014, 0.78828, 0.7822, 0.7873, 0.78025]
max |fresh OOF - packaged oof_A| = 0.0


## Final model (loaded) — live predictions

In [4]:
# LOADED final model_A.pkl (trained by run_all.py) -> live predictions vs scored table
model = joblib.load(C.MODEL_DIR/'model_A.pkl')
assert list(model.booster_.feature_name()) == list(X.columns)
p = model.predict_proba(X)[:, 1]
print('model_A: full-matrix live predict max |diff| vs pred_A =', float(np.max(np.abs(p - sc['pred_A'].to_numpy()))))
demo = sc.sample(5, random_state=1).assign(live_pred=lambda d: model.predict_proba(X.loc[d.index])[:, 1])
display(demo[['SK_ID_CURR','TARGET','pred_A','live_pred']])
del X; import gc; gc.collect()

model_A: full-matrix live predict max |diff| vs pred_A = 0.0


8